# E1 — per-slot teacher-forced rank sweep on supra50m (GPU)

**Question (phase-9 re-test).** Phase 9 concluded the deep interior of a real LLM is
'incompressible': one shared exemplar + a light per-slot LoRA, trained *end-to-end by
logit-KL*, failed to reproduce the model (held-out KL ~3.5). That test was **data-starved**
(307K params vs ~3K tokens) and trained through 8 simultaneously-corrupted slots. The leap
PR=6.7 (function-space) => 'weight correction must be high-rank' is also unjustified.

E1 decouples the question with **teacher forcing**: for each interior slot *i*, fit a rank-*r*
LoRA delta on ONE exemplar layer's weights to map the model's OWN true hidden state
`h_i -> h_{i+1}` (dense, plenty of data, no compounding). Sweep *r*, then **compose** all
fitted slots and measure end-to-end KL. Controls: rank-0 (gains only) floor, full-refit
ceiling, cascade fit (error-correcting), exemplar choice.

**Setup.** Just enable a **GPU** accelerator + **Internet** and Run All — the notebook
**auto-downloads the exact phase-9 model** (`SupraLabs/Supra-50M-Instruct`, a 12-layer
Llama, == local supra50m) via `huggingface_hub`. No `transformers` needed (a minimal CUDA
Llama forward is inlined; it is config-driven, so it also loads any standard Llama checkpoint
-- set `MODEL_ID` or add one as a Kaggle dataset). Exemplar/interior are derived from depth.

In [ ]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import glob, math, torch, torch.nn.functional as F
from safetensors.torch import load_file

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)

MODEL_ID = 'SupraLabs/Supra-50M-Instruct'   # the exact phase-9 model (== local supra50m)

def resolve_model():
    """1) env CKPT_DIR (CPU smoke), 2) a Kaggle dataset, 3) download MODEL_ID from HF."""
    env = os.environ.get('CKPT_DIR')
    if env and os.path.exists(os.path.join(env, 'config.json')): return env
    ds = glob.glob('/kaggle/input/**/config.json', recursive=True)
    if ds: return os.path.dirname(ds[0])
    from huggingface_hub import snapshot_download    # pre-installed on Kaggle; needs internet ON
    return snapshot_download(MODEL_ID, allow_patterns=['config.json', 'model.safetensors',
                                                       'tokenizer.json', 'tokenizer_config.json'])

MODEL_DIR = resolve_model()
RANKS    = [0, 1, 4, 16, 64]  # 0 = gains-only floor; 'full' refit added separately
N_SEQS   = 120                # teacher-forcing probe sequences (varied temperature)
SEQ_LEN  = 128
FIT_STEPS = 500
EVAL_SEQS = 24                # small subset for end-to-end KL (bounds GPU memory)
print('device', DEV, '| model', MODEL_DIR)

In [ ]:
def rotate_half(x):
    d = x.shape[-1] // 2
    return torch.cat([-x[..., d:], x[..., :d]], dim=-1)

class MinLlama:
    def __init__(self, path, dev=DEV):
        import json
        cfg = json.load(open(os.path.join(path, 'config.json')))
        self.H = cfg['num_attention_heads']
        self.KV = cfg.get('num_key_value_heads', self.H)
        self.d = cfg['hidden_size']
        self.hd = cfg.get('head_dim', self.d // self.H)
        self.L, self.eps = cfg['num_hidden_layers'], cfg.get('rms_norm_eps', 1e-5)
        rp = cfg.get('rope_parameters') or {}      # supra50m nests it; SmolLM has rope_theta top-level
        self.theta = cfg.get('rope_theta', rp.get('rope_theta', 10000))
        self.vocab = cfg['vocab_size']
        sd = load_file(os.path.join(path, 'model.safetensors'))
        self.w = {k: v.float().to(dev) for k, v in sd.items()}
        self.embed = self.w['model.embed_tokens.weight']
        self.head = self.w.get('lm_head.weight', self.embed)   # tied head if no lm_head.weight
        self.dev = dev
    def rms(self, x, g): return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * g
    def rope(self, x, pos):
        inv = 1.0 / (self.theta ** (torch.arange(0, self.hd, 2, device=self.dev).float() / self.hd))
        fr = pos[:, None].float() * inv[None, :]
        cos = torch.cat([fr.cos(), fr.cos()], -1)[None, None]
        sin = torch.cat([fr.sin(), fr.sin()], -1)[None, None]
        return x * cos + rotate_half(x) * sin
    def block(self, x, l, pos, W=None, gin=None, gpost=None):
        """Differentiable block. W: dict of the 7 proj weights (defaults to layer l's own,
        frozen); gin/gpost: RMSNorm gains (default layer l's)."""
        g = lambda n: (W[n] if W and n in W else self.w[f'model.layers.{l}.{n}.weight'])
        gin = self.w[f'model.layers.{l}.input_layernorm.weight'] if gin is None else gin
        gpost = self.w[f'model.layers.{l}.post_attention_layernorm.weight'] if gpost is None else gpost
        B, T, _ = x.shape
        h = self.rms(x, gin)
        q = (h @ g('self_attn.q_proj').T).view(B, T, self.H, self.hd).transpose(1, 2)
        k = (h @ g('self_attn.k_proj').T).view(B, T, self.KV, self.hd).transpose(1, 2)
        v = (h @ g('self_attn.v_proj').T).view(B, T, self.KV, self.hd).transpose(1, 2)
        q, k = self.rope(q, pos), self.rope(k, pos)
        rep = self.H // self.KV
        k, v = k.repeat_interleave(rep, 1), v.repeat_interleave(rep, 1)
        att = (q @ k.transpose(-1, -2)) / (self.hd ** 0.5)
        att = att + torch.full((T, T), float('-inf'), device=self.dev).triu(1)
        o = (att.softmax(-1) @ v).transpose(1, 2).reshape(B, T, self.d)
        x = x + o @ g('self_attn.o_proj').T
        h2 = self.rms(x, gpost)
        mlp = (F.silu(h2 @ g('mlp.gate_proj').T) * (h2 @ g('mlp.up_proj').T)) @ g('mlp.down_proj').T
        return x + mlp
    @torch.no_grad()
    def forward(self, idx, capture=False, slot_fn=None):
        """slot_fn(l, x, pos) -> x overrides block l (used to compose fitted adapters)."""
        pos = torch.arange(idx.shape[1], device=self.dev)
        x = self.embed[idx]; hs = [x]
        for l in range(self.L):
            x = slot_fn(l, x, pos) if slot_fn else self.block(x, l, pos)
            hs.append(x)
        logits = self.rms(x, self.w['model.norm.weight']) @ self.head.T
        return (logits, hs) if capture else logits
    @torch.no_grad()
    def generate(self, idx, n, temperature=0.9, top_k=40):
        for _ in range(n):
            lo = self.forward(idx[:, -256:])[:, -1, :] / temperature
            v, _ = torch.topk(lo, top_k); lo[lo < v[:, [-1]]] = float('-inf')
            idx = torch.cat([idx, torch.multinomial(lo.softmax(-1), 1)], 1)
        return idx

m = MinLlama(MODEL_DIR)
EXEMPLAR = m.L // 2                  # shared exemplar = a middle layer (L6 for supra50m)
INTERIOR = list(range(3, m.L - 1))   # redundant interior (phase-9: slots 3..10 for L=12)
print(f'loaded: L={m.L} d={m.d} H={m.H}/{m.KV} vocab={m.vocab}  tied_head={m.head is m.embed}')
print(f'exemplar=L{EXEMPLAR}  interior slots {INTERIOR[0]}..{INTERIOR[-1]} ({len(INTERIOR)})')

In [ ]:
# sanity: coherent generation + low entropy => forward is correct
bos = torch.tensor([[1]], device=DEV)
out = m.generate(bos, 40, temperature=0.7)
ent = (lambda lo: -(lo.softmax(-1) * lo.log_softmax(-1)).sum(-1).mean().item())(m.forward(out))
print(f'next-token entropy {ent:.2f} nats (uniform {math.log(m.vocab):.2f})')

In [ ]:
# ---- collect TRUE teacher-forcing hidden states on a varied-temperature probe ----
seqs = []
for t in (0.7, 1.0, 1.3):
    for _ in range(N_SEQS // 3):
        seqs.append(m.generate(torch.tensor([[1]], device=DEV), SEQ_LEN, temperature=t)[:, 1:])
idx = torch.cat(seqs, 0)
_, HS = m.forward(idx, capture=True)             # keep hidden states for fitting; drop big logits
HS = [h.detach() for h in HS]                    # HS[l] = input to block l ; HS[l+1] = its output
EVAL = idx[:EVAL_SEQS]                            # small subset for end-to-end KL
full_eval = m.forward(EVAL).detach()
print('probe:', tuple(idx.shape), '| eval subset:', tuple(EVAL.shape), '| hidden states:', len(HS))

In [ ]:
# ---- per-slot teacher-forced fit: map TRUE h_i -> TRUE h_{i+1} using exemplar + LoRA ----
PROJ = ('self_attn.q_proj','self_attn.k_proj','self_attn.v_proj','self_attn.o_proj',
        'mlp.gate_proj','mlp.up_proj','mlp.down_proj')

class SlotAdapter(torch.nn.Module):
    def __init__(self, exemplar, rank, full=False):
        super().__init__()
        self.full, self.rank, self.e = full, rank, exemplar
        self.A, self.B, self.Wfull = {}, {}, {}
        for n in PROJ:
            W = m.w[f'model.layers.{exemplar}.{n}.weight']
            if full:
                self.Wfull[n] = torch.nn.Parameter(W.clone())
            elif rank > 0:
                self.A[n] = torch.nn.Parameter(torch.randn(rank, W.shape[1], device=DEV) * 0.02)
                self.B[n] = torch.nn.Parameter(torch.zeros(W.shape[0], rank, device=DEV))
        for n in self.A:  # register_parameter forbids '.' in names
            self.register_parameter('A_' + n.replace('.', '_'), self.A[n])
            self.register_parameter('B_' + n.replace('.', '_'), self.B[n])
        for n in self.Wfull: self.register_parameter('W_' + n.replace('.', '_'), self.Wfull[n])
        self.gin = torch.nn.Parameter(m.w[f'model.layers.{exemplar}.input_layernorm.weight'].clone())
        self.gpost = torch.nn.Parameter(m.w[f'model.layers.{exemplar}.post_attention_layernorm.weight'].clone())
    def W(self):
        out = {}
        for n in PROJ:
            W = m.w[f'model.layers.{self.e}.{n}.weight']
            if self.full: out[n] = self.Wfull[n]
            elif self.rank > 0: out[n] = W + self.B[n] @ self.A[n]
            else: out[n] = W
        return out
    def apply(self, x, pos):
        return m.block(x, self.e, pos, W=self.W(), gin=self.gin, gpost=self.gpost)

def fit_slot(i, rank, full=False, h_in=None, steps=FIT_STEPS, bs=2048):
    """Fit slot i's adapter so block(exemplar)(h_in) ~= TRUE h_{i+1}. h_in defaults to TRUE h_i
    (teacher forcing); pass a cascade input to error-correct."""
    pos = torch.arange(idx.shape[1], device=DEV)
    src = HS[i] if h_in is None else h_in
    tgt = HS[i + 1]
    X = src.reshape(-1, m.d); Y = tgt.reshape(-1, m.d)
    true_upd = (Y - X)
    ad = SlotAdapter(EXEMPLAR, rank, full).to(DEV)
    opt = torch.optim.Adam(ad.parameters(), lr=3e-3)
    Bsz, T = src.shape[0], src.shape[1]
    for s in range(steps):
        bi = torch.randint(0, Bsz, (max(1, bs // T),), device=DEV)
        xb, yb = src[bi], tgt[bi]
        out = ad.apply(xb, pos)
        loss = (out - yb).pow(2).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        pred = ad.apply(src, pos).reshape(-1, m.d)
        rel = ((pred - X - true_upd).norm() / (true_upd.norm() + 1e-9)).item()
    return ad, rel

In [ ]:
# ---- RANK SWEEP: relative update error per (slot, rank) under teacher forcing ----
print('relative update error  ||pred_update - true_update|| / ||true_update||  (lower=better)')
print('slot |  ' + '  '.join(f'r={r:>2}' for r in RANKS) + '   full')
adapters = {}
for i in INTERIOR:
    row, best = [], {}
    for r in RANKS:
        ad, rel = fit_slot(i, r); row.append(rel); best[r] = ad
    adf, relf = fit_slot(i, 0, full=True)
    adapters[i] = best
    adapters[i]['full'] = adf
    print(f'{i:>4} |  ' + '  '.join(f'{x:4.2f}' for x in row) + f'   {relf:4.2f}')

In [ ]:
# ---- COMPOSE fitted adapters into the full model -> end-to-end KL (the real test) ----
def kl(p, q, chunk=2048):                         # chunked over tokens -> bounded memory
    p2, q2 = p.reshape(-1, p.shape[-1]), q.reshape(-1, q.shape[-1])
    tot, n = 0.0, p2.shape[0]
    for i in range(0, n, chunk):
        lp = p2[i:i+chunk].log_softmax(-1); lq = q2[i:i+chunk].log_softmax(-1)
        tot += (lp.exp() * (lp - lq)).sum(-1).sum().item()
    return tot / n

def compose_kl(rank_key):
    sel = {i: adapters[i][rank_key] for i in INTERIOR}
    def slot_fn(l, x, pos):
        return sel[l].apply(x, pos) if l in sel else m.block(x, l, pos)
    with torch.no_grad():
        lo = m.forward(EVAL, slot_fn=slot_fn)
    return kl(full_eval, lo)

# verbatim baseline: exemplar copied into the interior, no adapter
def verbatim_slot(l, x, pos): return m.block(x, EXEMPLAR, pos) if l in INTERIOR else m.block(x, l, pos)
with torch.no_grad():
    lo_v = m.forward(EVAL, slot_fn=verbatim_slot)
print(f'verbatim tile (no adapter) end-to-end KL = {kl(full_eval, lo_v):.3f}   [exact=0]')
for rk in RANKS + ['full']:
    print(f'composed rank={str(rk):>4}: end-to-end KL = {compose_kl(rk):.3f}')

In [ ]:
# ---- CASCADE fit (error-correcting): fit slot i on inputs propagated through the
# already-adapted earlier interior slots, target still TRUE h_{i+1}. Then compose. ----
casc = {}
@torch.no_grad()
def prop_to(i, rank_key):
    """Run the real model up to slot i but with interior slots <i replaced by casc adapters."""
    pos = torch.arange(idx.shape[1], device=DEV)
    x = m.embed[idx]
    for l in range(i):
        x = casc[l].apply(x, pos) if l in casc else m.block(x, l, pos)
    return x

RK = 16 if 16 in RANKS else RANKS[-1]
for i in INTERIOR:
    h_in = prop_to(i, RK) if i > INTERIOR[0] else HS[i]
    ad, _ = fit_slot(i, RK, h_in=h_in)
    casc[i] = ad
def casc_slot(l, x, pos): return casc[l].apply(x, pos) if l in casc else m.block(x, l, pos)
with torch.no_grad():
    lo_c = m.forward(EVAL, slot_fn=casc_slot)
print(f'cascade-fit (rank {RK}) end-to-end KL = {kl(full_eval, lo_c):.3f}')
# prefix-composition curve: adapt slots 3..i (rank RK, independent fit), own weights elsewhere
print('prefix-composition KL (adapt interior up to slot i):')
for i in INTERIOR:
    sel = {j: adapters[j][RK] for j in INTERIOR if j <= i}
    sf = lambda l, x, pos: sel[l].apply(x, pos) if l in sel else m.block(x, l, pos)
    with torch.no_grad(): loi = m.forward(EVAL, slot_fn=sf)
    print(f'  up to slot {i}: KL {kl(full_eval, loi):.3f}')

## How to read

- **Rank-sweep table** — the per-slot teacher-forced relative update error. If it drops to
  near-0 at small *r* (e.g. r=4–16), each deep slot IS a low-rank correction of the exemplar
  *locally* — the phase-9 'high-rank' inference (F2) was unjustified. Compare against `r=0`
  (gains-only floor) and `full` (unconstrained refit ceiling).
- **Composed end-to-end KL** — the real test. If composing low-rank adapters reaches ~0 KL
  (vs verbatim ~6.5), the deep interior IS compressible to exemplar + light per-slot maps and
  phase-9's verdict is overturned. If composed KL stays high even when per-slot errors are low,
  the obstacle is **composition covariate shift**, not rank — and the **cascade-fit** number
  plus the **prefix curve** localize where compounding ignites and how much error-correction
  recovers.
- **full-refit composed KL** is the ceiling: if even unconstrained per-slot refits fail to
  compose, no rank helps and the limit is dynamical, not representational.